# 01 — Bronze: Café Enchancia
**User story:** As a data engineer, I want the original store records preserved so that I can trace and reprocess sales.

This notebook reads one selected store's CSV files from a Unity Catalog Volume and saves a Delta table. Prices, discounts, currencies, local timestamps and order statuses remain unchanged. Both COMPLETED and CANCELLED orders are kept. No currency conversion, timezone conversion, deduplication or sales aggregation occurs here.

## Before running
- Upload `KR_SEOUL_20261019_20261025.csv` into `/Volumes/ws_tip_demo/tip_demo_schema/korea` for the default Korea run.
- Attach your assigned Databricks Spark compute. You need USE CATALOG, USE SCHEMA, READ VOLUME and CREATE TABLE permissions, plus permission to overwrite the target on reruns.
- Run the widget cell, review the textbox values, then run all cells. After changing a textbox, rerun the notebook from the configuration cell onward.
- Instructor default destination: `ws_tip_demo.tip_demo_schema.ec_bronze_kr_seoul`.

**Rerun behavior:** the selected store's Bronze table is replaced with all matching files currently in the source folder. Keep the entire intended workshop batch there. Other stores' tables are unaffected. This is a fixed-batch lab, not an incremental ingestion job. Students running the same store in the same schema share a target; use instructor-assigned group schemas for isolated group work.


## Workshop setup and scope
Set **store** and **target_schema** before running, then run the configuration cell and all following cells. Use the same values across Bronze, Silver and Gold. The instructor default schema is `tip_demo_schema`; students use their assigned `tip_group_01` through `tip_group_10`. Schemas and permissions must be provisioned before class. Source files remain in the shared training Volumes, regardless of the target schema.

Run Bronze, then Silver, then Gold for one assigned store. Gold `ALL` is an instructor consolidation option after all seven stores have been loaded in the same target schema. A one-store run creates three tables; a full seven-store run creates 15 tables: seven Bronze, seven Silver, one Gold.

This is a fixed seven-day archive exercise. No direct POS, email or network-drive connection is made by students. Scheduling the 8 a.m. New York report, arrival/completeness monitoring, alerts, incremental history, corrected-file handling and refunds remain follow-up backlog items. Tax is excluded in the fictional data. All original files for the selected store must remain present for a full-batch rerun.


In [ ]:
# Textbox widgets. Keep the same "store" parameter name in later layer notebooks.
dbutils.widgets.text("store", "KR_SEOUL", "Store ID (Korea: KR_SEOUL)")
dbutils.widgets.text("source_path", "/Volumes/ws_tip_demo/tip_demo_schema", "Base source path (country folder added automatically)")

dbutils.widgets.text("target_schema", "tip_demo_schema", "Target schema (use your assigned group schema)")


## 1. Read the configuration
The store textbox selects a store ID. `korea` and `seoul` are accepted aliases for `KR_SEOUL`. Other supported IDs are PH_LUZON, PH_VIS, PH_MIND, US_NY, US_QUEENS and US_NJ.

For another store, change **store**. Keep **source_path** at the base `/Volumes/ws_tip_demo/tip_demo_schema`; the code adds `korea`, `philippines`, or `usa`. Do not append the country folder yourself. Change the base only if the instructor provides another source location. Filenames must start with the store ID followed by `_`, as in the supplied dataset. The destination table is separate for each store.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

STORE_IDS = {"PH_LUZON", "PH_VIS", "PH_MIND", "US_NY", "US_QUEENS", "US_NJ", "KR_SEOUL"}

# Map each store ID to its Volume subfolder name.
STORE_VOLUME_MAP = {
    "KR_SEOUL":   "korea",
    "PH_LUZON":   "philippines",
    "PH_VIS":     "philippines",
    "PH_MIND":    "philippines",
    "US_NY":      "usa",
    "US_QUEENS":  "usa",
    "US_NJ":      "usa",
}

store_input = dbutils.widgets.get("store").strip().upper()
STORE = {"KOREA": "KR_SEOUL", "SEOUL": "KR_SEOUL"}.get(store_input, store_input)
if STORE not in STORE_IDS:
    raise ValueError(f"Unknown store {STORE!r}. Use one of: {sorted(STORE_IDS)}")

# Build SOURCE_PATH dynamically from the base volume path + store-specific subfolder.
BASE_VOLUME = dbutils.widgets.get("source_path").strip().rstrip("/")
if not BASE_VOLUME.startswith("/Volumes/"):
    raise ValueError("source_path must be a Unity Catalog /Volumes/ folder.")
SOURCE_PATH = f"{BASE_VOLUME}/{STORE_VOLUME_MAP[STORE]}"

# Select the instructor-provisioned target_schema widget for isolated group work.
CATALOG = "ws_tip_demo"
SCHEMA = dbutils.widgets.get("target_schema").strip()
import re
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError("target_schema must be a valid assigned schema name.")
if not spark.catalog.databaseExists(f"{CATALOG}.{SCHEMA}"):
    raise ValueError(f"Schema {CATALOG}.{SCHEMA} is unavailable. Ask your instructor to provision it.")
TARGET_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`ec_bronze_{STORE.lower()}`"
spark.conf.set("spark.sql.session.timeZone", "UTC")  # For ingestion metadata only.
print(f"Store: {STORE}\nSource: {SOURCE_PATH}\nTarget: {TARGET_TABLE}")


## 2. Find the selected store's files
Read matching CSVs directly inside the source folder. The notebook stops before writing if there are no matching files. Reference CSVs such as the menu are not selected. For the supplied Korea fixture, one file contains seven local days.


In [ ]:
source_files = sorted(
    item.path for item in dbutils.fs.ls(SOURCE_PATH)
    if item.name.startswith(f"{STORE}_") and item.name.lower().endswith(".csv")
)
if not source_files:
    raise ValueError(f"No {STORE}_*.csv files found directly in {SOURCE_PATH}. Upload the store CSV first.")
for path in source_files:
    print(path)


## 3. Read raw records and add metadata
All 14 business columns remain strings. Enforcing the expected header contract stops files with missing or reordered columns from being silently loaded by position. Three extra fields record the filename, source path and ingestion timestamp. `ingested_at_utc` is load time, not transaction time or the time an email arrived.


In [ ]:
SOURCE_COLUMNS = [
    "store_id", "store_name", "country", "time_zone", "currency",
    "business_date", "timestamp_local", "transaction_id", "order_status",
    "product_name", "category", "quantity", "unit_price", "discount_amount"
]
source_schema = StructType([StructField(name, StringType(), True) for name in SOURCE_COLUMNS])
bronze = (
    spark.read.schema(source_schema)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .csv(source_files)
    .select("*",
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_path").alias("source_path"))
    .withColumn("ingested_at_utc", F.current_timestamp())
)
# Reading all columns here also triggers CSV/header validation before the write.
display(bronze.limit(10))


## 4. Check the batch before writing
Bronze checks scope and record preservation. It does not discard business records. A filename that contains another store's records is rejected as a delivery mistake, rather than silently filtering those records out.

**Acceptance criteria:** every source row is preserved, all 14 source fields remain strings, metadata is populated, and only the selected store's destination is written. Completed and cancelled orders must both remain available when present.


In [ ]:
source_count = bronze.count()
if source_count == 0:
    raise ValueError("The selected files contain no data rows. Nothing has been overwritten.")
wrong_store = bronze.filter(F.col("store_id").isNull() | (F.col("store_id") != STORE)).count()
if wrong_store:
    raise ValueError(f"{wrong_store} rows have a missing or different store_id. Check the source files.")
assert all(isinstance(bronze.schema[name].dataType, StringType) for name in SOURCE_COLUMNS)
assert bronze.columns == SOURCE_COLUMNS + ["source_file", "source_path", "ingested_at_utc"]
print(f"Rows to preserve: {source_count:,}")
display(bronze.groupBy("store_id", "currency", "order_status").count())


## 5. Save the Bronze Delta table
Overwrite the selected store's workshop table to prevent reruns from appending the same batch again. Keep source files stable while the notebook is running. No source files are changed or deleted.


In [ ]:
(bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE))
print(f"Saved {TARGET_TABLE}")


## 6. Verify the saved result
Compare the persisted rows with the source using all business fields plus source filename/path, including duplicate multiplicities. Ingestion timestamps are excluded from that comparison because they are generated at load time. This proves that Bronze preserved values, rather than merely reaching the same count.

Run the notebook again with unchanged files. Counts and business values should remain the same, while ingestion timestamps can change.


In [ ]:
saved = spark.table(TARGET_TABLE)
assert saved.count() == source_count, "Source and Bronze row counts differ."
compare_columns = SOURCE_COLUMNS + ["source_file", "source_path"]
source_values = bronze.select(*compare_columns)
saved_values = saved.select(*compare_columns)
assert source_values.exceptAll(saved_values).limit(1).count() == 0, "Source records missing from Bronze."
assert saved_values.exceptAll(source_values).limit(1).count() == 0, "Unexpected records in Bronze."
assert saved.filter(F.col("ingested_at_utc").isNull()).count() == 0
assert all(isinstance(saved.schema[name].dataType, StringType) for name in SOURCE_COLUMNS)
display(saved.groupBy("business_date", "currency", "order_status").count().orderBy("business_date", "order_status"))
print(f"Bronze checks passed: {source_count:,} rows preserved in {TARGET_TABLE}.")
